# pyCAFE — Direct Frequency Sweep Example

This notebook walks through a **direct (forced harmonic) acoustic analysis** of a 2D rectangular cavity.

Steps:
1. Create the mesh with Gmsh
2. Load the mesh and inspect boundary names
3. **Select boundary conditions interactively** (one dropdown per boundary)
4. Set the frequency sweep range
5. Assemble the FEM system and run the sweep
6. Visualise the pressure field at a selected frequency (matplotlib)
7. Visualise the pressure field inline with **pyvista**
8. Extract and plot a point frequency response function (FRF)
9. *(Optional)* Export results to VTK / ParaView

---
**Geometry:** rectangular 2D cavity  
**Element type:** CQUAD8 (serendipity quadrilateral, order 2)  
**Fluid:** air at 20 °C

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import tempfile, pathlib

import gmsh
import pycafe
from pycafe.solver.solver_helmholtz_1 import solve_helmholtz_frequency_sweep
from pycafe.build_matrices.assembly_cquad8 import expand_to_full
from pycafe.post_processing.post_processing import weighted_pressure_at_point

## Step 1 — Geometry, mesh, and fluid parameters

In [ ]:
# ── Cavity dimensions ─────────────────────────────────────────────────────────
Lx = 1.0    # length [m]
Ly = 0.5    # height [m]

# ── Fluid properties ──────────────────────────────────────────────────────────
rho = 1.204   # density [kg/m³]
c0  = 343.0   # speed of sound [m/s]

# ── Mesh ─────────────────────────────────────────────────────────────────────
h     = 0.07  # target element size [m]
order = 2     # 1 = CQUAD4,  2 = CQUAD8

## Step 2 — Generate and load the mesh

In [ ]:
def make_rect_mesh(Lx, Ly, h, order, filepath):
    """Structured transfinite quad mesh for a rectangle."""
    try:
        if gmsh.isInitialized():
            gmsh.finalize()
    except Exception:
        pass
    gmsh.initialize()
    gmsh.option.setNumber("General.Verbosity", 0)
    gmsh.model.add("cavity")
    p1 = gmsh.model.geo.addPoint(0,  0,  0, h)
    p2 = gmsh.model.geo.addPoint(Lx, 0,  0, h)
    p3 = gmsh.model.geo.addPoint(Lx, Ly, 0, h)
    p4 = gmsh.model.geo.addPoint(0,  Ly, 0, h)
    l1 = gmsh.model.geo.addLine(p1, p2)
    l2 = gmsh.model.geo.addLine(p2, p3)
    l3 = gmsh.model.geo.addLine(p3, p4)
    l4 = gmsh.model.geo.addLine(p4, p1)
    cl   = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
    surf = gmsh.model.geo.addPlaneSurface([cl])
    gmsh.model.addPhysicalGroup(1, [l1], name="bottom")
    gmsh.model.addPhysicalGroup(1, [l2], name="right")
    gmsh.model.addPhysicalGroup(1, [l3], name="top")
    gmsh.model.addPhysicalGroup(1, [l4], name="left")
    gmsh.model.addPhysicalGroup(2, [surf], name="domain")
    gmsh.model.geo.synchronize()
    nx = max(int(round(Lx / h)) + 1, 3)
    ny = max(int(round(Ly / h)) + 1, 3)
    for line in [l1, l3]:
        gmsh.model.mesh.setTransfiniteCurve(line, nx)
    for line in [l2, l4]:
        gmsh.model.mesh.setTransfiniteCurve(line, ny)
    gmsh.model.mesh.setTransfiniteSurface(surf)
    gmsh.option.setNumber("Mesh.SecondOrderIncomplete", 1)
    gmsh.model.mesh.generate(2)
    gmsh.model.mesh.recombine()
    gmsh.model.mesh.setOrder(order)
    gmsh.write(str(filepath))
    gmsh.finalize()

_msh_file = pathlib.Path(tempfile.mktemp(suffix=".msh"))
make_rect_mesh(Lx, Ly, h, order, _msh_file)
nodes, elements, boundaries = pycafe.load_mesh(str(_msh_file), show_info=True, show_plot=True)
_msh_file.unlink(missing_ok=True)

print(f"\nMesh loaded: {nodes.shape[0]} nodes")
print(f"Boundaries available: {list(boundaries.keys())}")

## Step 3 — Select boundary conditions

| Option | Description |
|--------|-------------|
| **Hard wall** | Rigid wall — zero normal velocity (default Neumann BC) |
| **Zero pressure** | Pressure constrained to p = 0 Pa |
| **Constant pressure** | Prescribed constant pressure amplitude [Pa] |
| **Impedance** | Acoustic impedance Z (complex) [Pa·s/m] |
| **Normal velocity** | Prescribed normal velocity amplitude [m/s] |

> **Tip:** a typical forced-response setup uses **Normal velocity** on one wall (piston source) and **Hard wall** on the remaining three.

In [ ]:
BC_OPTIONS = [
    "Hard wall (rigid)",
    "Zero pressure",
    "Constant pressure",
    "Impedance",
    "Normal velocity",
]

_bc_dropdowns = {}
_bc_rows = []

for bname in boundaries:
    if bname == "domain":
        continue
    default = "Normal velocity" if bname == "left" else "Hard wall (rigid)"
    dd = widgets.Dropdown(
        options=BC_OPTIONS,
        value=default,
        description=f"{bname}:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="380px"),
    )
    _bc_dropdowns[bname] = dd
    _bc_rows.append(dd)

_extra_label    = widgets.HTML("<b>Extra parameters (used only for the selected BC types):</b>")
_w_pressure_val = widgets.FloatText(value=1.0,   description="Pressure [Pa]:",
                                     style={"description_width": "150px"}, layout=widgets.Layout(width="300px"))
_w_imp_real     = widgets.FloatText(value=415.0, description="Z real [Pa·s/m]:",
                                     style={"description_width": "150px"}, layout=widgets.Layout(width="300px"))
_w_imp_imag     = widgets.FloatText(value=0.0,   description="Z imag [Pa·s/m]:",
                                     style={"description_width": "150px"}, layout=widgets.Layout(width="300px"))
_w_velocity     = widgets.FloatText(value=1.0,   description="v_n [m/s]:",
                                     style={"description_width": "150px"}, layout=widgets.Layout(width="300px"))

display(
    widgets.VBox(_bc_rows + [
        widgets.HTML("<hr>"),
        _extra_label,
        _w_pressure_val,
        _w_imp_real, _w_imp_imag,
        _w_velocity,
    ])
)

In [ ]:
bc_pressure_zero      = []
bc_pressure_constant  = []
bc_impedance          = []
bc_velocity_walls     = []

for bname, dd in _bc_dropdowns.items():
    choice = dd.value
    if choice == "Zero pressure":
        bc_pressure_zero.append(bname)
    elif choice == "Constant pressure":
        bc_pressure_constant.append(bname)
    elif choice == "Impedance":
        bc_impedance.append(bname)
    elif choice == "Normal velocity":
        bc_velocity_walls.append(bname)

bc = (
    bc_pressure_zero,
    bc_pressure_constant,
    float(_w_pressure_val.value),
    bc_impedance,
    complex(_w_imp_real.value, _w_imp_imag.value),
    bc_velocity_walls,
    float(_w_velocity.value),
    None,
    0.0,
)

print("Boundary conditions summary")
print(f"  Hard wall     : {[b for b,d in _bc_dropdowns.items() if d.value == 'Hard wall (rigid)']}")
print(f"  Zero pressure : {bc_pressure_zero}")
print(f"  Const. pressure ({_w_pressure_val.value} Pa) : {bc_pressure_constant}")
print(f"  Impedance ({_w_imp_real.value}+{_w_imp_imag.value}j Pa·s/m) : {bc_impedance}")
print(f"  Normal velocity ({_w_velocity.value} m/s) : {bc_velocity_walls}")

## Step 4 — Set the frequency sweep range

In [ ]:
_w_fmin = widgets.FloatText(value=10.0,  description="f_min [Hz]:",
                             style={"description_width": "120px"}, layout=widgets.Layout(width="260px"))
_w_fmax = widgets.FloatText(value=600.0, description="f_max [Hz]:",
                             style={"description_width": "120px"}, layout=widgets.Layout(width="260px"))
_w_df   = widgets.FloatText(value=5.0,   description="Δf [Hz]:",
                             style={"description_width": "120px"}, layout=widgets.Layout(width="260px"))
display(widgets.VBox([_w_fmin, _w_fmax, _w_df]))

## Step 5 — Assemble the FEM system and run the frequency sweep

In [ ]:
frequencies = np.arange(
    float(_w_fmin.value),
    float(_w_fmax.value) + float(_w_df.value),
    float(_w_df.value),
)
print(f"Frequency sweep: {frequencies[0]:.1f} – {frequencies[-1]:.1f} Hz  ({len(frequencies)} points)")

system = pycafe.prepare_acoustic_system(
    nodes=nodes, elements=elements, boundaries=boundaries,
    rho=rho, c0=c0, bc=bc, debug=False,
)
print(f"System assembled. Reduced DOFs: {system['K_red'].shape[0]}")

P_red = solve_helmholtz_frequency_sweep(
    K_red=system["K_red"],
    M_red=system["M_red"],
    C_red=system["C_red"],
    frequencies=frequencies,
    pressure_nodes_red=system["pressure_nodes_red"],
    pressure_values=system["pressure_values"],
    nodes=nodes,
    boundary_velocity_nodes=system["bc_velocity"],
    idx_free=system["idx_free"],
    rho=rho,
    v_n=system["value_velocity_normal"],
    boundaries=boundaries,
    elements=elements,
)

p_full = expand_to_full(
    P_red, system["idx_free"], system["p0_nodes"], nodes.shape[0]
)
print(f"Sweep complete. p_full shape: {p_full.shape}  (nodes × frequencies)")

## Step 6 — Pressure field at a selected frequency (matplotlib)

Use the slider to step through frequencies.

In [ ]:
def plot_field(freq_index):
    i = freq_index - 1
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sc0 = axes[0].scatter(nodes[:, 0], nodes[:, 1], c=np.real(p_full[:, i]),
                          cmap="RdBu_r", s=20)
    plt.colorbar(sc0, ax=axes[0], label="Re(p) [Pa]")
    axes[0].set_title(f"Real part — f = {frequencies[i]:.1f} Hz")
    sc1 = axes[1].scatter(nodes[:, 0], nodes[:, 1], c=np.abs(p_full[:, i]),
                          cmap="hot_r", s=20)
    plt.colorbar(sc1, ax=axes[1], label="|p| [Pa]")
    axes[1].set_title(f"Magnitude — f = {frequencies[i]:.1f} Hz")
    for ax in axes:
        ax.set_aspect("equal")
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        ax.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_field,
    freq_index=widgets.IntSlider(
        value=1, min=1, max=len(frequencies), step=1,
        description="Freq idx:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="500px"),
    ),
);

## Step 7 — Visualise pressure field inline with pyvista

The cell below renders the acoustic pressure field on the actual FEM mesh — elements and edges visible.  
Change `FREQ_TO_SHOW` to inspect any frequency in the sweep.

In [ ]:
import pyvista as pv

pv.set_jupyter_backend("trame")  # interactive inline rendering

# ── Build pyvista UnstructuredGrid from pyCAFE arrays ────────────────────────
pts3d = nodes.copy().astype(np.float64)
if pts3d.shape[1] == 2:
    pts3d = np.column_stack([pts3d, np.zeros(len(pts3d))])

_quad_key = next(
    k for k in elements
    if "Quadrilateral" in k or "Quadrangle" in k or "quad" in k.lower()
)
conn       = elements[_quad_key].astype(np.int64) - 1
n_cells, n_per_cell = conn.shape
vtk_type   = 23 if n_per_cell == 8 else 9

_cells_flat = np.hstack(
    [np.full((n_cells, 1), n_per_cell, dtype=np.int64), conn]
).ravel()
_cell_types = np.full(n_cells, vtk_type, dtype=np.uint8)

pv_grid = pv.UnstructuredGrid(_cells_flat, _cell_types, pts3d)
print(f"pyvista grid: {pv_grid.n_points} points, {pv_grid.n_cells} cells")

In [ ]:
# ── Choose frequency to display ───────────────────────────────────────────────
FREQ_TO_SHOW = 100.0   # Hz  ← change to any value in your sweep range

i = int(np.argmin(np.abs(frequencies - FREQ_TO_SHOW)))
print(f"Showing f = {frequencies[i]:.1f} Hz  (index {i})")

pv_grid.point_data["pressure_abs"]  = np.abs(p_full[:, i]).astype(np.float64)
pv_grid.point_data["pressure_real"] = np.real(p_full[:, i]).astype(np.float64)

# ── Two-panel plot: magnitude and real part ───────────────────────────────────
plotter = pv.Plotter(shape=(1, 2), notebook=True)

plotter.subplot(0, 0)
plotter.add_mesh(
    pv_grid, scalars="pressure_abs", cmap="hot_r",
    show_edges=True, edge_color="grey",
    scalar_bar_args={"title": "|p| [Pa]", "n_labels": 4},
)
plotter.add_title(f"|p|  —  {frequencies[i]:.1f} Hz", font_size=11)
plotter.view_xy()

plotter.subplot(0, 1)
plotter.add_mesh(
    pv_grid, scalars="pressure_real", cmap="RdBu_r",
    show_edges=True, edge_color="grey",
    scalar_bar_args={"title": "Re(p) [Pa]", "n_labels": 4},
)
plotter.add_title(f"Re(p)  —  {frequencies[i]:.1f} Hz", font_size=11)
plotter.view_xy()

plotter.show()

## Step 8 — Point frequency response function (FRF)

Enter the coordinates of a probe point. Pressure is interpolated using the three nearest nodes (inverse-distance weighting).

In [ ]:
_w_px = widgets.FloatText(value=Lx/2, description="Probe x [m]:",
                           style={"description_width": "130px"}, layout=widgets.Layout(width="280px"))
_w_py = widgets.FloatText(value=Ly/2, description="Probe y [m]:",
                           style={"description_width": "130px"}, layout=widgets.Layout(width="280px"))
display(widgets.HBox([_w_px, _w_py]))

In [ ]:
probe_xy = np.array([float(_w_px.value), float(_w_py.value)])
p_point  = weighted_pressure_at_point(p_full, nodes[:, :2], probe_xy, num_closest=3)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.semilogy(frequencies, np.abs(p_point), color="#2c5f8a", linewidth=1.8)
ax1.set_ylabel("|p| [Pa]", fontweight="bold")
ax1.set_title(f"Pressure FRF at ({probe_xy[0]:.3f}, {probe_xy[1]:.3f}) m", fontweight="bold")
ax1.grid(True, which="both", linewidth=0.4)

ax2.plot(frequencies, np.degrees(np.angle(p_point)), color="#b82020", linewidth=1.8)
ax2.set_ylabel("Phase [°]", fontweight="bold")
ax2.set_xlabel("Frequency [Hz]", fontweight="bold")
ax2.set_yticks([-180, -90, 0, 90, 180])
ax2.grid(True, linewidth=0.4)

plt.tight_layout()
plt.show()

data = np.column_stack([
    frequencies, np.real(p_point), np.imag(p_point),
    np.abs(p_point), np.angle(p_point),
])
header = (
    "Frequency[Hz]    Re(p)[Pa]    Im(p)[Pa]    |p|[Pa]    Phase[rad]\n"
    f"Probe: x={probe_xy[0]:.4f} m, y={probe_xy[1]:.4f} m"
)
np.savetxt("frf_probe.txt", data, header=header, fmt="%.6e")
print("FRF saved to frf_probe.txt")

## Step 9 — (Optional) Export to VTK / ParaView

Writes one `.vtu` per frequency + a `.pvd` collection.  
Open `vtu_pressure/pressure_results.pvd` in ParaView — the time slider steps through frequencies.

In [ ]:
from pycafe.post_processing.export_vtk import export_pressure_vtu

pvd_path = export_pressure_vtu(
    nodes=nodes,
    elements=elements,
    p_full=p_full,
    frequencies=frequencies,
    output_dir="vtu_pressure",
)
print(f"Open in ParaView: {pvd_path}")